# Lab 5 — FeatureTools Auto Feature Engineering

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Build a FeatureTools **EntitySet** from the loans table.
2. Run **Deep Feature Synthesis (DFS)** with `max_depth=1`.
3. Inspect the engineered **feature matrix** shape and column names.
4. Rank auto-generated features by **|correlation|** with `default`.

> **Checkpoints:** shape **(1000, 6)** · **6** engineered features · top corr = **int_rate** (~0.21)

<!-- cisco-day04-lab05-expanded-2026 -->

**Data:** **1000** loans from `lending_club_sample.csv`

**Day 4 flow:** Labs 1–4 KNN & API → **Lab 5 (you are here)** FeatureTools → Lab 6 MLflow + DVC.

## Deep Feature Synthesis (DFS) in one slide

FeatureTools automates feature engineering by applying **primitives** (sum, mean, count, etc.) across entities in an **EntitySet**.

| Concept | This lab |
|---------|----------|
| **EntitySet** | One table: `loans` indexed by `loan_id` |
| **DFS** | `ft.dfs(..., max_depth=1)` — one hop of transformations |
| **Output** | `feature_matrix` (1000 rows) + `feature_defs` (column recipes) |

## FeatureTools vs hand-picked features (Day 3)

| Approach | Pros | Cons |
|----------|------|------|
| Hand-picked | Interpretable, fast | Misses interactions |
| DFS | Discovers transforms automatically | Needs review before production |

---

## 1. Load loans and select DFS columns

In [ ]:
from pathlib import Path

import pandas as pd

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

import featuretools as ft
from IPython.display import display

dfs_cols = [
    "loan_id",
    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "installment",
    "default",
]

raw = pd.read_csv(LENDING_CLUB_CSV)
raw["default"] = raw["loan_status"].isin(DEFAULT_STATUSES).astype(int)
df = raw[dfs_cols].copy()
df["loan_id"] = df["loan_id"].astype(str)

print(f"input columns: {len(df.columns)}")
print(f"rows: {len(df)}")
display(df.head(3))


`loan_id` becomes the **index** for the entity. `default` stays in the frame so DFS can include it in the output matrix for correlation analysis.

### 1b. Input column dtypes

In [ ]:
print(df.dtypes)
print(f"\nnull counts:\n{df.isna().sum()}")


### 1c. Default rate in DFS frame

In [ ]:
print(f"default rate: {df['default'].mean():.4f}")


---

## 2. Build the EntitySet

In [ ]:
es = ft.EntitySet(id="lending")
es = es.add_dataframe(
    dataframe_name="loans",
    dataframe=df,
    index="loan_id",
)

print(es)


### 2b. EntitySet structure

In [ ]:
print("dataframes:", list(es.dataframes.keys()))
print("loans shape:", es["loans"].shape)


---

## 3. Run Deep Feature Synthesis

In [ ]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="loans",
    max_depth=1,
    verbose=False,
)

print("Lab 5 — FeatureTools auto FE")
print(f"engineered features: {len(feature_defs)}")
print(f"feature matrix shape: {feature_matrix.shape}")
display(feature_matrix.head(3))


### 3b. Column names in feature matrix

In [ ]:
print("columns:", list(feature_matrix.columns))


---

## 4. Inspect feature definitions

In [ ]:
def_names = [fd.get_name() for fd in feature_defs]
defs_df = pd.DataFrame({"feature": def_names})
display(defs_df)


Each `feature_def` records the **primitive** and input columns — useful for reproducing or auditing auto-generated features in production.

### 4b. Primitive types used

In [ ]:
primitives = [type(fd).__name__ for fd in feature_defs]
print(pd.Series(primitives).value_counts())


---

## 5. Correlation with default

In [ ]:
numeric_cols = [
    c for c in feature_matrix.columns
    if c != "default" and str(feature_matrix[c].dtype) != "category"
]
top_corr = (
    feature_matrix[numeric_cols + ["default"]]
    .corr(numeric_only=True)["default"]
    .drop("default", errors="ignore")
    .abs()
    .sort_values(ascending=False)
)

print("top |corr| with default (first 3):")
for name, val in top_corr.head(3).items():
    print(f"  {name}: {val:.4f}")

display(top_corr.head(5).to_frame("abs_corr").round(4))


**int_rate** leads — consistent with Day 3 logistic regression where `int_rate` had the strongest positive coefficient.

### 5b. Signed correlation for top feature

In [ ]:
signed = (
    feature_matrix[numeric_cols + ["default"]]
    .corr(numeric_only=True)["default"]
    .drop("default", errors="ignore")
)
print(f"int_rate signed corr: {signed['int_rate']:.4f}")


---

## 6. Compare to hand-picked Day 3 features

In [ ]:
hand_picked = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]
hand_corr = (
    raw[hand_picked + ["default"]]
    .corr(numeric_only=True)["default"]
    .drop("default")
    .abs()
    .sort_values(ascending=False)
)

compare = pd.DataFrame({
    "source": ["DFS top", "Hand-picked top"],
    "feature": [top_corr.index[0], hand_corr.index[0]],
    "abs_corr": [top_corr.iloc[0], hand_corr.iloc[0]],
})
display(compare.round(4))


DFS surfaces the same signal domain experts chose — but on richer multi-table data it can discover interactions you'd miss manually.

### 6b. Full hand-picked ranking

In [ ]:
display(hand_corr.to_frame("abs_corr").round(4))


---

## 7. Feature matrix statistics

In [ ]:
display(feature_matrix.describe().round(2))


### 7b. Any missing values after DFS?

In [ ]:
print(feature_matrix.isna().sum().sum(), "total nulls")


---

## 8. What changes with `max_depth=2`?

With a single table, `max_depth=2` may add little. With **related tables** (borrowers, payments), deeper DFS creates cross-entity aggregations — the real power of FeatureTools.

### 8b. Row count sanity check

In [ ]:
assert len(feature_matrix) == len(df) == 1000
print("DFS preserved all loan rows:", len(feature_matrix))


### 8c. Feature matrix dtypes

In [ ]:
print(feature_matrix.dtypes)


### 8d. Scatter — int_rate vs default (top feature)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(feature_matrix["int_rate"], feature_matrix["default"], alpha=0.25, s=10)
ax.set_xlabel("int_rate")
ax.set_ylabel("default")
ax.set_title("Engineered int_rate vs default")
plt.tight_layout()
plt.show()


### 8e. Save feature matrix (optional artifact)

In [ ]:
# feature_matrix.to_parquet("dfs_features.parquet")
print("Feature matrix ready for downstream modeling pipelines.")


---

## 9. Feature selection preview

Top correlated features are candidates for KNN or logistic models — always review for leakage.

In [ ]:
top3 = top_corr.head(3).index.tolist()
print("Top 3 DFS features by |corr|:", top3)
subset = feature_matrix[top3 + ["default"]]
display(subset.corr(numeric_only=True).round(4))


### 9b. Drop default from modeling matrix

In [ ]:
X_dfs = feature_matrix.drop(columns=["default"], errors="ignore")
print("Modeling matrix shape (no target):", X_dfs.shape)


### 9c. Feature variance check

In [ ]:
variance = X_dfs.var().sort_values(ascending=False)
display(variance.round(2).to_frame("variance"))


---

## 10. Try it yourself

Which engineered feature has the **second** highest |corr| with default?

In [ ]:
# Your code here (optional)


In [ ]:
second = top_corr.index[1]
print(f"Second highest |corr|: {second} = {top_corr.iloc[1]:.4f}")


---

## 11. Checkpoint summary

In [ ]:
assert len(df.columns) == 7
assert len(feature_defs) == 6
assert feature_matrix.shape == (1000, 6)
assert top_corr.index[0] == "int_rate"
assert abs(top_corr.iloc[0] - 0.2084) < 0.02
print("✓ All checkpoint assertions passed")


---

## Reflection

1. What new features might appear with `max_depth=2` or multiple related tables?
2. Why filter out categorical columns before correlation ranking?
3. Would you feed the DFS matrix directly into KNN without reviewing primitives?

**Previous:** [Lab 4 — FastAPI scoring API](lab04_fastapi_scoring_api.ipynb)  
**Next:** [Lab 6 — MLflow experiment log](lab06_mlflow_experiment_log.ipynb)